# Устранение размытия изображений

![blur.png](attachment:blur.png)

В этом задании курса вам предстоит реализовать метод устранения размытия изображений (image deblurring).
Мы начнем с простейшей возможной архитектуры нейросети, а к концу задания вы реализуете современную SOTA-архитектуру для устранения размытия.

In [6]:
#!/bin/bash
curl -L -o ./parts-damages.zip https://www.kaggle.com/api/v1/datasets/download/humansintheloop/car-parts-and-car-damages

unzip ./parts-damages.zip 

SyntaxError: invalid syntax (914300096.py, line 2)

# Общая часть (15 баллов)

## Установка зависимостей

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

WSD = '/content/drive/My Drive/Colab Notebooks/debluring'

import sys
sys.path.append(WSD)

In [ ]:
from IPython.display import clear_output

!pip install torch torchvision
!pip install tqdm requests scikit-image matplotlib albumentations

clear_output()

## Загрузка датасета

In [ ]:
from utils import test_model
import os
import requests
from tqdm import tqdm

def download_file(session, url, save_path):
    response = session.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    with open(save_path, 'wb') as file:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=save_path) as pbar:
            for data in response.iter_content(chunk_size=8192):
                file.write(data)
                pbar.update(len(data))

TARGET_DIR = WSD
FILENAME = "GoPro.zip"

if not os.path.exists(os.path.join(TARGET_DIR, FILENAME)):
    print("Начинается загрузка датасета (около 5.5 ГБ). Пожалуйста, подождите...")
    session = requests.Session()
    file_url = 'https://titan.gml-team.ru:5003/fsdownload/zFoCt5HPQ/GoPro.zip'
    session.get(file_url)
    download_file(session, file_url, os.path.join(TARGET_DIR, FILENAME))

In [ ]:
# !unzip -qq '/content/drive/My Drive/Colab Notebooks/debluring/GoPro.zip' -d '/content/drive/My Drive/Colab Notebooks/debluring'

In [ ]:
!ls '/content/drive/My Drive/Colab Notebooks/debluring' -all

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'axes.titlesize': 'small'})

import torch
import torch.nn.functional as F

import torchvision
from torchvision.utils import make_grid

from torchvision import transforms
import os

from tqdm import tqdm

print(f"GPU: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

### Подготовка датасета

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path

class GoProDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        blur_path = Path(f"{root_dir}/blur")
        sharp_path = Path(f"{root_dir}/sharp")
        self.blurry_images = sorted([p.name for p in list(blur_path.glob("*.png"))])
        self.sharp_images = sorted([p.name for p in list(sharp_path.glob("*.png"))])

    def __len__(self):
        return len(self.blurry_images)

    def __getitem__(self, idx):
        blurry_path = Path(self.root_dir, 'blur', self.blurry_images[idx])
        sharp_path = Path(self.root_dir, 'sharp', self.sharp_images[idx])

        blurry_image = np.array(Image.open(blurry_path).convert('RGB'))
        sharp_image = np.array(Image.open(sharp_path).convert('RGB'))

        if self.transform:
            blurry_image = blurry_image.astype(np.float32)/255.0
            sharp_image = sharp_image.astype(np.float32)/255.0
            augmented = self.transform(image=blurry_image, image1=sharp_image)
            blurry_image = augmented['image']
            blurry_image = blurry_image.to(torch.float32)
            sharp_image = augmented['image1']
            sharp_image = sharp_image.to(torch.float32)

        return blurry_image, sharp_image

transform = A.Compose([
    A.RandomCrop(256, 256, p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    ToTensorV2(),
], additional_targets={
        'image1': 'image'}, p=1)

transform_test = A.Compose([
    ToTensorV2(),
], additional_targets={
        'image1': 'image'}, p=1)

PATH_TO_TRAIN = os.path.join(WSD, 'GoPro/train')
PATH_TO_TEST = os.path.join(WSD, 'GoPro/test')

train_dataset = GoProDataset(root_dir=PATH_TO_TRAIN, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=3, shuffle=True, num_workers=16) # Для ускорения процесса обучения можно попробовать увеличить num_workers

test_dataset = GoProDataset(root_dir=PATH_TO_TEST, transform=transform_test)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

## Baseline

## Базовая нейросеть (1 балл)
Мы начнем с реализации простейшей возможной сверточной нейронной сети.
Несколько сверточных слоев без пулинга (pooling).

Этот подход напоминает ранние работы в области [Super-Resolution](https://arxiv.org/abs/1501.00092) и [Denoising](https://arxiv.org/abs/1608.03981)

**Важно отметить, что наша сеть будет обучаться предсказывать остаточное значение (residual) для устранения размытия**

$I_{deblur}=I_{input} + f_{\theta}(I_{input})$

In [ ]:
# GRADED CELL: Baseline
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBnBlock(nn.Module):
    def __init__(self, in_channels, out_channels, ksize=3):
        """Простой сверточный блок

        Ваша задача - реализовать следующие модули:

            conv + bn + relu

        """
        super().__init__()

        # Your code vvv
        self.c1 = nn.Conv2d(in_channels, out_channels, kernel_size=ksize, padding=(ksize - 1)//2, padding_mode='reflect')
        self.bn = nn.BatchNorm2d(out_channels)
        # Your code ^^^

    def forward(self, x):
        # Your code vvv
        x = F.relu(self.bn(self.c1(x)))

        return x
        # Your code ^^^


class Baseline(nn.Module):
    def __init__(self, block, n_blocks=5, n_filters=64, ksize=5):
        """Базовая сверточная модель

        Ваша задача - реализовать следующую архитектуру:

            входной слой свертки (3 -> n_filters)
            n_blocks слоев ConvBnBlock
            выходной слой свертки (n_filters -> 3)

        """

        super().__init__()
        # Your code vvv

        self.c1 = nn.Conv2d(3, n_filters, kernel_size=ksize, stride=1, padding=(ksize - 1)//2, padding_mode='reflect', bias=True)
        self.bn1 = nn.BatchNorm2d(n_filters)
        self.blocks = nn.ModuleList([block(n_filters, n_filters, ksize=ksize) for _ in range(n_blocks)])
        self.c2 = nn.Conv2d(n_filters, 3, kernel_size=ksize, stride=1, padding=(ksize - 1)//2, padding_mode='reflect', bias=True)
        self.bn2 = nn.BatchNorm2d(3)

        # Your code ^^^

    def forward(self, x):
        inp = x
        # Your code vvv
        inp = self.c1(inp)
        inp = self.bn1(inp)

        for block_ in self.blocks:
            inp = block_(inp)

        inp = self.c2(inp)
        inp = self.bn2(inp)

        # Your code ^^^
        return x + inp

In [ ]:
baseline = Baseline(ConvBnBlock)
test_input = torch.rand(1, 3, 256, 256)
test_output = baseline(test_input)
assert test_input.shape == test_output.shape

### Функция потерь PSNR (0.5 балла)

Мы будем напрямую оптимизировать метрику качества PSNR вместо обычной MSE:

$PSNR = 10 * \log_{10}(\frac{MAX_I^2}{MSE})$

В нашем случае изображения нормализованы в диапазон [0, 1], поэтому $MAX_I=1$

Будьте внимательны при усреднении: сначала вычислите MSE для каждой пары изображений, затем примените логарифм, и только после этого усредните по батчу.

In [ ]:
# GRADED CELL: PSNRLoss
import torch
import torch.nn as nn
import torch.nn.functional as F

class PSNRLoss(nn.Module):
    def __init__(self):
        """Функция потерь Peak signal-to-noise ratio

        ПРИМЕЧАНИЕ: во время обучения мы минимизируем функцию потерь, но большее значение PSNR означает лучшее качество
        Поэтому вы можете внести -1 в логарифм при реализации,
        тем самым избавившись от необходимости деления
        """
        super().__init__()
        self.eps = 1e-8  # используем eps чтобы избежать 0 в логарифме

    def forward(self, pred, target):
        """
            Реализуйте следующее вычисление:

                10 * mean(log10(mse(pred, target)))

        """
        # Your code vvv
        mse = F.mse_loss(pred, target)

        log_mse = torch.log10(mse + self.eps)

        mean_log_mse = torch.mean(log_mse)

        return 10 * mean_log_mse
        # Your code ^^^

In [ ]:
criterion = PSNRLoss()
a = torch.tensor([[[[0.1632, 0.0024, 0.9913, 0.8892],
          [0.5655, 0.4472, 0.4592, 0.2013],
          [0.7722, 0.9089, 0.1708, 0.3654],
          [0.6147, 0.9567, 0.7018, 0.2376]]]])
b = torch.tensor([[[[0.8498, 0.1168, 0.3987, 0.6781],
          [0.7864, 0.9762, 0.3694, 0.9926],
          [0.9000, 0.0293, 0.0454, 0.0984],
          [0.9478, 0.3730, 0.9617, 0.5052]]]])
assert torch.isclose(criterion(a, b), torch.tensor(-6.8417))

### Обучение (0.75 балла)
Базовый процесс обучения.

Обратите внимание на использование gradient clipping (ограничение градиентов). Хотя для обучения простейшей модели это не обязательно, это значительно поможет в последующих частях задания!

Подсказка: используйте [`torch.nn.utils.clip_grad_norm_`](https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html), значение параметра `max_norm=0.05` хорошо работает в данной задаче.

In [ ]:
# GRADED CELL: train_model
def train_model(model, train_dataloader, optimizer, criterion, scheduler, num_epochs=200, checkpoints_path="./checkpoints", use_grad_clip=True, save_checkpoints=True, load_last_checkpoint=False):
    # Создаем папку для чекпоинтов
    if save_checkpoints:
        PATH = os.path.join(WSD, checkpoints_path)
        SAVE_EPOCH = 5  # Сохраняем каждые 5 эпох
        os.makedirs(PATH, exist_ok = True)

    last_epoch = -1
    last_dict = None
    if load_last_checkpoint:
        for file_name in tqdm(list(os.listdir(PATH))):
            try:
                cur_dict = torch.load(os.path.join(PATH, file_name), map_location=torch.device('cpu'))
                if cur_dict['epoch'] > last_epoch:
                    last_epoch = cur_dict['epoch']
                    if last_dict is not None:
                        del last_dict

                    last_dict = cur_dict
                else:
                    del cur_dict
            except:
                pass

        if last_dict is not None:
            model.load_state_dict(last_dict['model_state_dict'])
            optimizer.load_state_dict(last_dict['optimizer_state_dict'])
            scheduler.load_state_dict(last_dict['scheduler_state_dict'])
            last_epoch = last_dict['epoch']
            print(f'Загружен последний чекпоинт из эпохи {last_epoch}')
            del last_dict

    model.train()

    # Цикл обучения
    for epoch in range(num_epochs):

        loss_epoch = 0

        for inputs, targets in tqdm(train_dataloader, desc=f'Эпоха {epoch+1}'):
            # Your code vvv

            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            loss.backward()

            if use_grad_clip:
                # Здесь нужно реализовать ограничение градиентов
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.05)

            optimizer.step()

            # Your code ^^^


            loss_epoch += loss.item()

        # Сохраняем чекпоинт каждые SAVE_EPOCH эпох
        if save_checkpoints and epoch % SAVE_EPOCH == 0:
            torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    }, os.path.join(PATH, f"epoch_{epoch+1}.pth"))

        # Обновляем learning rate
        scheduler.step()
        print(f'Эпоха {epoch+1}, Loss: {loss.item()}, Acg_Epoch_Loss: {loss_epoch / len(train_dataloader)}, LR: {scheduler.get_last_lr()[0]}')

In [ ]:
from utils import get_scheduler

torch.manual_seed(11)

model = Baseline(ConvBnBlock)
model = model.to(device)

criterion = PSNRLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
use_grad_clip = True
scheduler = get_scheduler(optimizer)

train_model(model, train_dataloader, optimizer, criterion, scheduler, num_epochs=100, checkpoints_path='./checkpoints/baseline')

In [ ]:
# Сохраним веса модели в файл - его (или файл из чекпоинта) нужно будет сдать в тестирующую систему
baseline_model_path = os.path.join(WSD,'./baseline_model.pth')
torch.save({
        'model_state_dict': model.state_dict()
        }, baseline_model_path)

In [ ]:
# Загрузим веса модели из файла - проверим, что в тестирующей системе все правильно загрузится
model = Baseline(ConvBnBlock)
baseline_model_path = os.path.join(WSD,'./baseline_model.pth')
model.load_state_dict(torch.load(baseline_model_path, map_location=torch.device('cpu'))['model_state_dict'])
model = model.to(device)

In [ ]:
# Протестируем нашу модель
result = test_model(model, device, test_dataloader)
assert result <= 0.00252
print("Поздравляем!")

In [ ]:
print(result)

In [ ]:
pics = test_dataset[60]
blurred, gt = pics[0], pics[1]
plt.figure(figsize=(20, 4))
plt.suptitle("Визуальное сравнение")
plt.subplot(131)
plt.title("Размытое")
plt.imshow(blurred.permute(1, 2, 0))
plt.subplot(132)
plt.title("Результат модели")
with torch.no_grad():
    output = model(blurred.unsqueeze(0).to(device)).cpu().squeeze(0).permute(1, 2, 0).numpy()
output = np.clip(output, 0, 1)
plt.imshow(output)
plt.subplot(133)
plt.imshow(gt.permute(1, 2, 0))
plt.title("Оригинал")
plt.show()

## UNet

Следующим шагом будет обработка изображения на нескольких масштабах, подобно многомасштабным методам классического компьютерного зрения.

Наиболее распространенная модель, использующая несколько масштабов - это [UNet](https://arxiv.org/abs/1505.04597), впервые предложенная для медицинской сегментации и содержащая много необычных архитектурных решений.  
Тем не менее, она доказала свою эффективность во многих задачах компьютерного зрения и широко используется в задачах восстановления изображений и видео.

U-Net получили такое название из-за их U-образной формы, где входное изображение сначала уменьшается в размерности в нисходящей части (encoder), а затем увеличивается обратно до исходного размера в восходящей части (decoder).

![unet.png](attachment:unet.png)

UNet может быть реализован либо с конкатенацией skip-connections (пропускных соединений), либо с их суммированием.  
Результаты обычно не сильно отличаются друг от друга.

### Двойная свертка (0.5 балла)
В UNet каждый сверточный блок состоит из двух последовательных сверток.  
Вам нужно реализовать простой стек из двух сверточных слоев с двумя ReLU активациями.

In [ ]:
# GRADED CELL: UNetBlock


class UNetBlock(nn.Module):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class ConvBnBlock(nn.Module):
        def __init__(self, in_channels, out_channels, ksize=3):
            """Простой сверточный блок

            Ваша задача - реализовать следующие модули:

                conv + bn + relu

            """
            super().__init__()

            # Your code vvv
            self.c1 = nn.Conv2d(in_channels, out_channels, kernel_size=ksize, padding=(ksize - 1)//2, padding_mode='reflect')
            self.bn = nn.BatchNorm2d(out_channels)
            # Your code ^^^

        def forward(self, x):
            # Your code vvv
            x = F.relu(self.bn(self.c1(x)))

            return x
            # Your code ^^^

    def __init__(self, in_channels, ksize=3):
        """Базовый блок архитектуры UNet

        Ваша задача - реализовать следующие модули:

            conv + bn + relu + conv + bn + relu

        """
        super().__init__()
        # Your code vvv
        self.blocks = nn.ModuleList([ConvBnBlock(in_channels, in_channels, ksize=ksize) for _ in range(2)])
        # Your code ^^^

    def forward(self, x):
        # Your code vvv
        for block in self.blocks:
            x = block(x)

        return x
        # Your code ^^^

In [ ]:
block = UNetBlock(64)
vec = torch.rand(16, 64, 256, 256)
assert block(vec).shape == torch.Size([16, 64, 256, 256])

### UNet Down block (0.5 балла)

In [ ]:
# GRADED CELL: UNetDownBlock
class UNetDownBlock(nn.Module):
    def __init__(self, chan):
        """Блок понижающей дискретизации (downsampling) в энкодере UNet

            Ваша задача - реализовать следующие модули:

                AvgPool + Conv 1x1

            Пространственная размерность входа **уменьшается** в 2 раза
            Количество каналов **увеличивается** в 2 раза
        """
        super().__init__()
        # Your code vvv

        self.s_conv = nn.Conv2d(chan, 2 * chan, kernel_size=1, bias=True)
        self.s_avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

        # Your code ^^^

    def forward(self, x):
        # Your code vvv

        x = self.s_conv(x)

        x = self.s_avg_pool(x)

        return x
        # Your code ^^^

In [ ]:
block = UNetDownBlock(64)
vec = torch.rand(16, 64, 256, 256)
assert block(vec).shape == torch.Size([16, 128, 128, 128])

### UNet Up block (0.5 балла)

In [ ]:
# GRADED CELL: UNetUpBlock
class UNetUpBlock(nn.Module):
    def __init__(self, chan):
        """Блок повышающей дискретизации (upsampling) в декодере UNet

            Ваша задача - реализовать следующие модули:

                Upsample + Conv 1x1

            Пространственная размерность входа **увеличивается** в 2 раза
            Количество каналов **уменьшается** в 2 раза
        """
        super().__init__()
        # Your code vvv
        self.s_up = nn.Upsample(scale_factor=2, mode='nearest')
        self.s_conv = nn.Conv2d(chan, chan // 2, kernel_size=1, bias=True)
        # Your code ^^^

    def forward(self, x):
        # Your code vvv
        x = self.s_up(x)

        x = self.s_conv(x)

        return x
        # Your code ^^^

In [ ]:
block = UNetUpBlock(64)
vec = torch.rand(16, 64, 256, 256)
assert block(vec).shape == torch.Size([16, 32, 512, 512])

### Обобщенная архитектура UNet (3 балла)
За прошедшие годы архитектура UNet доказала свою эффективность.  
Многие последующие работы сохраняют архитектуру на макроуровне и изменяют только отдельные блоки, например: меняют операцию понижающей дискретизации, изменяют количество сверток в блоке, добавляют механизм внимания в узкое место сети (bottleneck) и т.д.

Поэтому мы также реализуем обобщенную архитектуру.

**Чтобы получить баллы за эту часть, ваша модель должна пройти проверку качества**

In [ ]:
# GRADED CELL: GeneralizedUNet

class GeneralizedUNet(nn.Module):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    def __init__(self, block, downBlock, upBlock, img_channel=3, width=32,
                 middle_blk_num=1, enc_blk_nums=[1,1,1,1], dec_blk_nums=[1,1,1,1]):
        """Обобщенная архитектура UNet

            Эта часть уже реализована за вас
            Но вы можете изменить ее при необходимости

        """
        super().__init__()

        if enc_blk_nums[-1] > 2:
            width = 20

        self.intro = nn.Conv2d(in_channels=img_channel, out_channels=width, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=1, bias=True)

        self.ending = nn.Conv2d(in_channels=width, out_channels=img_channel, kernel_size=1, stride=1, groups=1, bias=True)


        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.middle_blks = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()

        self.skip_conv = nn.ModuleList()


        chan = width

        for num in enc_blk_nums:
            self.encoders.append(
                nn.Sequential(
                    *[block(chan) for _ in range(num)]
                )
            )
            self.downs.append(
                downBlock(chan)
            )

            chan = chan * 2

        self.middle_blks = \
            nn.Sequential(
                *[block(chan) for _ in range(middle_blk_num)]
            )

        for num in dec_blk_nums:
            self.ups.append(
                upBlock(chan)
            )

            chan = chan // 2

            self.skip_conv.append(nn.Conv2d(in_channels=2 * chan, out_channels=chan, kernel_size=1, bias=True))


            self.decoders.append(
                nn.Sequential(
                    *[block(chan) for _ in range(num)]
                )
            )


    def forward(self, x):
        activations = []

        inp = x

        # Your code vvv
        x = self.intro(x)

        for encoder, down in zip(self.encoders, self.downs):
            x = encoder(x)
            activations.append(x)
            x = down(x)

        x = self.middle_blks(x)

        for decoder, up, skip, skip_conv_ in zip(self.decoders, self.ups, reversed(activations), self.skip_conv):
            x = up(x)
            x = skip_conv_(torch.cat((skip, x), dim=1))
            x = decoder(x)

        x = self.ending(x)

        return inp + x

        # Your code ^^^

In [ ]:
model = GeneralizedUNet(UNetBlock, UNetDownBlock, UNetUpBlock)
vec = torch.rand(16, 3, 256, 256)
assert model(vec).shape == torch.Size([16, 3, 256, 256])

### Обучаем UNet

In [ ]:
from utils import get_scheduler

torch.manual_seed(11)

model = GeneralizedUNet(UNetBlock, UNetDownBlock, UNetUpBlock)
model = model.to(device)

criterion = PSNRLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
use_grad_clip = True
scheduler = get_scheduler(optimizer)

train_model(model, train_dataloader, optimizer, criterion, scheduler, num_epochs=100,
            checkpoints_path='./checkpoints/unet', use_grad_clip=use_grad_clip)

In [ ]:
# Сохраним веса модели в файл - его (или файл из чекпоинта) нужно будет сдать в тестирующую систему
unet_model_path = os.path.join(WSD, './unet_model.pth')
torch.save({
        'model_state_dict': model.state_dict()
        }, unet_model_path)

In [ ]:
# Загрузим веса модели из файла - проверим, что в тестирующей системе все правильно загрузится
model = GeneralizedUNet(UNetBlock, UNetDownBlock, UNetUpBlock)
unet_model_path = os.path.join(WSD, './unet_model.pth')
model.load_state_dict(torch.load(unet_model_path, map_location=torch.device('cpu'))['model_state_dict'])
model = model.to(device)

In [ ]:
result = test_model(model, device, test_dataloader)
assert result <= 0.00235

In [ ]:
pics = test_dataset[2]
blurred, gt = pics[0], pics[1]
plt.figure(figsize=(20, 4))
plt.suptitle("Визуальное сравнение")
plt.subplot(131)
plt.title("Размытое")
plt.imshow(blurred.permute(1, 2, 0))
plt.subplot(132)
plt.title("Результат модели")
with torch.no_grad():
    output = model(blurred.unsqueeze(0).to(device)).cpu().squeeze(0).permute(1, 2, 0).numpy()
output = np.clip(output, 0, 1)
plt.imshow(output)
plt.subplot(133)
plt.imshow(gt.permute(1, 2, 0))
plt.title("Оригинал")
plt.show()

## NAFNet

Как обсуждалось в предыдущем разделе, многие сети строятся на основе UNet и изменяют отдельные блоки.
[NAFNet](https://arxiv.org/abs/2204.04676) долгое время являлся SOTA-подходом для устранения размытия, хотя и был представлен как базовая модель.

Основные отличия:
* Отсутствие стандартных нелинейностей (нет ReLU/GELU/ELU и др.)
* Упрощенный механизм channel attention
* Использование LayerNorm вместо BatchNorm
* И множество других приемов обучения, заимствованных из работ по трансформерам

В некотором смысле эта работа похожа на [ConvNeXt](https://arxiv.org/abs/2201.03545), где авторы также "стряхнули пыль" с ResNet и обучили SOTA CNN для классификации.

**И теперь ваша задача - реализовать NAFNet :)**

### Простой блок гейтинга (Simple gate) (0.5 балла)

![gate.png](attachment:gate.png)

Вместо использования обычных нелинейностей NAFNet предлагает использовать "Simple Gates" (простые гейты), которые выполняют поэлементное умножение карт признаков.

Слой Simple gate разделяет входную карту признаков на 2 части вдоль оси каналов и перемножает их.

Для реализации может пригодиться функция [`torch.chunk`](https://pytorch.org/docs/stable/generated/torch.chunk.html)

In [ ]:
# GRADED CELL: SimpleGate
class SimpleGate(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        """
            Разделить вход на 2 части вдоль оси каналов
            Вернуть произведение этих 2 частей
        """
        # Your code vvv
        c1, c2 = torch.chunk(x, 2, dim=1)

        return c1 * c2
        # Your code ^^^

In [ ]:
block = SimpleGate()
vec = torch.rand(16, 64, 256, 256)
assert block(vec).shape == torch.Size([16, 32, 256, 256])

### NAFNet up block (0.5 балла)

In [ ]:
# GRADED CELL: NAFNetUpBlock
class NAFNetUpBlock(nn.Module):
    def __init__(self, channels):
        """Блок повышающей дискретизации (upsampling) NAFNet

        Реализуйте и используйте следующие модули:

            свертка 1x1 (chan -> 2 * chan)
            pixelshuffle(2)

        """
        super().__init__()
        # Your code vvv
        self.s_conv = nn.Conv2d(channels, 2 * channels, kernel_size=1, bias=True)
        self.s_pixel_shuffle = nn.PixelShuffle(2)
        # Your code ^^^

    def forward(self, x):
        # Your code vvv
        x = self.s_conv(x)

        x = self.s_pixel_shuffle(x)

        return x
        # Your code ^^^

In [ ]:
block = NAFNetUpBlock(64)
vec = torch.rand(16, 64, 256, 256)
assert block(vec).shape == torch.Size([16, 32, 512, 512])

### NAFNet down block (0.5 балла)

In [ ]:
# GRADED CELL: NAFNetDownBlock
class NAFNetDownBlock(nn.Module):
    def __init__(self, channels):
        """Блок понижающей дискретизации (downsampling) NAFNet

        Реализуйте и используйте следующие модули:

            свертка с шагом (stride) 2, **обратите внимание на padding**

        """
        super().__init__()
        # Your code vvv
        self.s_conv = nn.Conv2d(channels, 2 * channels, kernel_size=3, stride=2, padding=1, padding_mode='reflect', bias=True)
        # Your code ^^^

    def forward(self, x):
        # Your code vvv
        x = self.s_conv(x)

        return x
        # Your code ^^^

In [ ]:
block = NAFNetDownBlock(64)
vec = torch.rand(16, 64, 256, 256)
print(block(vec).shape )
assert block(vec).shape == torch.Size([16, 128, 128, 128])

### Упрощенный механизм внимания по каналам (0.75 балла)

Обычный механизм channel attention создает веса для каждого канала входной карты признаков, используя 2-слойный MLP.  
Авторы NAFNet предлагают убрать малый MLP для каналов, просто применяя пулинг и линейное преобразование карты признаков для получения весов каналов.

![gate.png](attachment:gate.png)

In [ ]:
# GRADED CELL: SCA
class SCA(nn.Module):
    def __init__(self, in_channels, out_channels):
        """Упрощенный модуль channel attention

        Реализуйте и используйте следующие модули:

            adaptiveavgpool для получения карты признаков 1x1
            проекционный слой свертки 1x1

        """
        super().__init__()
        # Your code vvv
        self.s_adapt_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.s_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=True)
        # Your code ^^^

    def forward(self, x):
        """
            Вернуть только веса внимания (attention weights)

        """
        # Your code vvv
        x = self.s_adapt_avg_pool(x)

        x = self.s_conv(x)

        return x
        # Your code ^^^

In [ ]:
block = SCA(64, 32)
vec = torch.rand(16, 64, 256, 256)
assert block(vec).shape == torch.Size([16, 32, 1, 1])

### Блок NAFNet (6 баллов)

**Финальный босс этой части задания**.

На диаграмме показана внутренняя структура блока:

![nafnet_block.png](attachment:nafnet_block.png)

Обратите внимание на использование обучаемых коэффициентов масштабирования `beta` и `gamma` для skip-connection. Используйте [`nn.Parameter`](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html)

**Чтобы получить баллы за эту часть, ваша модель должна пройти проверку качества**

In [ ]:
# GRADED CELL: NAFNetBlock
from utils import LayerNorm2d  # use this layernorm

class NAFNetBlock(nn.Module):
    def __init__(self, c, DW_Expand=2, FFN_Expand=2):
        super().__init__()

        c_exp = c * DW_Expand
        ffn_channel = FFN_Expand * c

        self.beta = nn.Parameter(torch.randn(1,1))
        self.gamma = nn.Parameter(torch.randn(1,1))

        # Your code vvv
        self.layer_norm_left = LayerNorm2d(c)
        self.conv_left_1 = nn.Conv2d(c, c_exp, kernel_size=1, bias=True)
        self.conv_left_2 = nn.Conv2d(c_exp, c_exp, kernel_size=3, padding=1, padding_mode='reflect', bias=True)
        self.s_gate_left = SimpleGate()
        self.sca = SCA(c_exp // 2, c_exp // 2)

        self.conv_middle = nn.Conv2d(c_exp // 2, c, kernel_size=1, bias=True)

        self.layer_norm_right = LayerNorm2d(c)
        self.conv_right_1 = nn.Conv2d(c, ffn_channel, kernel_size=3, padding=1, padding_mode='reflect', bias=True)
        self.s_gate_right = SimpleGate()
        self.conv_right_2 = nn.Conv2d(ffn_channel // 2, c, kernel_size=3, padding=1, padding_mode='reflect', bias=True)

        # Your code ^^^

    def forward(self, inp):
        # Your code vvv

        left_norm = self.layer_norm_left(inp)

        left_gate = self.s_gate_left(self.conv_left_2(self.conv_left_1(left_norm)))

        left_sca = self.sca(left_gate)

        #-----------------------------------------------------------

        middle_conv = self.conv_middle(left_sca * left_gate)

        #-----------------------------------------------------------

        right_norm_inp = middle_conv * self.beta + inp

        right_norm = self.layer_norm_right(right_norm_inp)

        right_conv = self.conv_right_2(self.s_gate_right(self.conv_right_1(right_norm)))

        out = right_conv * self.beta + right_norm_inp

        return out
        # Your code ^^^

In [ ]:
block = NAFNetBlock(64)
vec = torch.rand(16, 64, 256, 256)
assert block(vec).shape == torch.Size([16, 64, 256, 256])

### Обучаем NAFNet

In [ ]:
from utils import get_scheduler

torch.manual_seed(11)

model = GeneralizedUNet(NAFNetBlock, NAFNetDownBlock, NAFNetUpBlock, enc_blk_nums=[1,2,2,28], dec_blk_nums=[2,2,2,1])
model = model.to(device)

criterion = PSNRLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
use_grad_clip = True
scheduler = get_scheduler(optimizer)

train_model(model, train_dataloader, optimizer, criterion, scheduler, checkpoints_path='./checkpoints/nafnet', use_grad_clip=use_grad_clip)

In [ ]:
# Сохраним веса модели в файл - его (или файл из чекпоинта) нужно будет сдать в тестирующую систему
nafnet_model_path = os.path.join(WSD,'./nafnet_model.pth')
torch.save({
        'model_state_dict': model.state_dict()
        }, nafnet_model_path)

In [ ]:
# Загрузим веса модели из файла - проверим, что в тестирующей системе все правильно загрузится
model = GeneralizedUNet(NAFNetBlock, NAFNetDownBlock, NAFNetUpBlock, enc_blk_nums=[1,2,2,28], dec_blk_nums=[2,2,2,1])
nafnet_model_path = os.path.join(WSD,'./nafnet_model.pth')

model.load_state_dict(torch.load(nafnet_model_path, map_location=torch.device('cpu'))['model_state_dict'])
model = model.to(device)

In [ ]:
result = test_model(model, device, test_dataloader)
assert result <= 0.002

## Результат работы модели

In [ ]:
pics = test_dataset[60]
blurred, gt = pics[0], pics[1]
plt.figure(figsize=(20, 4))
plt.suptitle("Визуальное сравнение")
plt.subplot(131)
plt.title("Размытое")
plt.imshow(blurred.permute(1, 2, 0))
plt.subplot(132)
plt.title("Результат модели")
with torch.no_grad():
    output = model(blurred.unsqueeze(0).to(device)).cpu().squeeze(0).permute(1, 2, 0).numpy()
output = np.clip(output, 0, 1)
plt.imshow(output)
plt.subplot(133)
plt.imshow(gt.permute(1, 2, 0))
plt.title("Оригинал")
plt.show()

In [ ]:
# Рассчитываем PSNR и SSIM между предсказанным изображением и оригиналом
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

gt_np = gt.permute(1, 2, 0).numpy()

psnr_value = peak_signal_noise_ratio(gt_np, output, data_range=1.0)
ssim_value = structural_similarity(gt_np, output, multichannel=True, channel_axis=2,data_range=1.0)
print("PSNR:", psnr_value, "SSIM:", ssim_value)

# Соревновательная часть (35 баллов)

В этой части вам нужно обучить свою классную модель деблюринга и побить решения остальных участников.<br><br>
Функция `get_my_cool_model` должна возвращать модель, загруженную из файла model_path, если его передали. Если его не передали, то можно обучить модель прямо внутри функции и вернуть.

In [ ]:
# GRADED CELL: get_my_cool_model

def get_my_cool_model(model_path=None, device=device):
    # Your code vvv
    class MyGeneralizedUNet(nn.Module):
        def __init__(self, block, downBlock, upBlock, img_channel=3, width=32,
                    middle_blk_num=(8, 8, 8), enc_blk_nums=[(4, 1, 1), (4, 1, 1), (6, 2, 2), (6, 4, 4)], dec_blk_nums=[(6, 4, 4), (6, 2, 2), (4, 1, 1),(4, 1, 1)]): # [(n_blocks, pWiseAttantionHeads, dWiseAttantionHeads), ...]

            super().__init__()

            self.intro = nn.Conv2d(in_channels=img_channel, out_channels=width, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=1, bias=True)
            n_blocks, pWiseHeads, dWiseHeads = enc_blk_nums[0]
            self.introTransformerBlock = nn.Sequential(*[block(width, pWiseHeads, dWiseHeads) for _ in range(n_blocks)])

            n_blocks, pWiseHeads, dWiseHeads = dec_blk_nums[-1]
            self.outroTransformerBlock = nn.Sequential(*[block(2 * width, pWiseHeads, dWiseHeads) for _ in range(n_blocks)])
            self.outro = nn.Conv2d(in_channels=2 * width, out_channels=img_channel, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=1, bias=False)


            self.encoders = nn.ModuleList()
            self.decoders = nn.ModuleList()
            self.middle_blks = nn.ModuleList()
            self.ups = nn.ModuleList()
            self.downs = nn.ModuleList()

            self.skip_conv = nn.ModuleList()


            chan = width

            for num in enc_blk_nums[1:]:
                self.encoders.append(
                    nn.Sequential(
                        *[block(chan, num[1], num[2]) for _ in range(num[0])]
                    )
                )
                self.downs.append(
                    downBlock(chan)
                )

                chan = chan * 2

            self.middle_blks = \
                nn.Sequential(
                    *[block(chan, middle_blk_num[1], middle_blk_num[2]) for _ in range(middle_blk_num[0])]
                )

            for num in dec_blk_nums[:-1]:
                self.ups.append(
                    upBlock(chan)
                )

                chan = chan // 2

                self.skip_conv.append(nn.Conv2d(in_channels=2 * chan, out_channels=chan, kernel_size=1, bias=True))


                self.decoders.append(
                    nn.Sequential(
                        *[block(chan, num[1], num[2]) for _ in range(num[0])]
                    )
                )


        def forward(self, x):
            activations = []

            inp = x

            # Your code vvv
            x = self.intro(x)
            x = self.introTransformerBlock(x)
            topSkip = x

            for encoder, down in zip(self.encoders, self.downs):
                x = encoder(x)
                activations.append(x)
                x = down(x)

            x = self.middle_blks(x)

            for decoder, up, skip, skip_conv_ in zip(self.decoders, self.ups, reversed(activations), self.skip_conv):
                x = up(x)
                x = skip_conv_(torch.cat((skip, x), dim=1))
                x = decoder(x)

            x = self.outroTransformerBlock(torch.cat((topSkip, x), dim=1))
            shift = self.outro(x)

            return inp + shift

            # Your code ^^^
############################################################################################################################################
    class DWiseAttention(nn.Module):
        def __init__(self, in_channels):

            super().__init__()
            # Your code vvv
            self.Norm = LayerNorm2d(in_channels)
            self.Qconv = nn.Conv2d(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=in_channels, bias=True)
            self.Kconv = nn.Conv2d(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=in_channels, bias=True)

            self.sm = nn.Softmax(dim=2)

            self.Vconv = nn.Conv2d(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=in_channels, bias=True)

            # Your code ^^^

        def forward(self, x):

            # Your code vvv
            norm = self.Norm(x)

            Q = self.Qconv(norm)
            K = self.Kconv(norm)
            V = self.Vconv(norm) # (b_size, C, H, W)

            b_size, C, H, W = Q.shape

            Q_reshaped = Q.view(b_size, C, H*W)
            K_reshaped = K.view(b_size, C, H*W)
            V_reshaped = V.view(b_size, C, H*W)

            attention = (Q_reshaped @ K_reshaped.transpose(1, 2)) / torch.sqrt(torch.tensor(H * W)) # (b_size, C, C)

            attention =  self.sm(attention)

            shift = attention @ V_reshaped
            shift = shift.view(b_size, C, H, W)

            return shift
            # Your code ^^^
############################################################################################################################################
    class PixelWiseAttention(nn.Module):
        def __init__(self, in_channels):

            super().__init__()
            # Your code vvv
            self.Norm1 = LayerNorm2d(in_channels)

            self.Qconv = nn.Conv2d(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=1, bias=True)
            self.Kconv = nn.Conv2d(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=1, bias=True)

            self.Norm2 = nn.BatchNorm2d(in_channels)

            self.Vconv = nn.Conv2d(in_channels=in_channels, out_channels=in_channels, kernel_size=1, stride=1, groups=1, bias=True)

            self.OutConv = nn.Conv2d(in_channels=in_channels, out_channels=in_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=in_channels, bias=False)

            # Your code ^^^

        def forward(self, x):

            # Your code vvv
            norm = self.Norm1(x)

            Q = self.Qconv(norm)
            K = self.Kconv(norm)
            V = self.Vconv(norm) # (b_size, C, H, W)

            attention = Q * K

            attention =  self.Norm2(attention)

            shift = V * attention
            shift = self.OutConv(shift)

            return shift
            # Your code ^^^
############################################################################################################################################
    class MultiHeadAttantion(nn.Module):
        def __init__(self, in_channels, pixelWiseHeads=1, dWiseHeads=1):

            super().__init__()
            # Your code vvv

            self.pAttention = nn.ModuleList([PixelWiseAttention(in_channels=in_channels) for _ in range(pixelWiseHeads)])
            self.dAttention = nn.ModuleList([DWiseAttention(in_channels=in_channels) for _ in range(dWiseHeads)])
            self.OutConv = nn.Conv2d(in_channels=in_channels * (pixelWiseHeads + dWiseHeads), out_channels=in_channels, kernel_size=1, stride=1, groups=1, bias=False)

            # Your code ^^^

        def forward(self, x):

            # Your code vvv
            shifts = []

            inp = x

            for pa in self.pAttention:
                shifts.append(pa(x))

            for da in self.dAttention:
                shifts.append(da(x))

            shift = self.OutConv(torch.cat(shifts, dim=1))

            return inp + shift
            # Your code ^^^
############################################################################################################################################
    class FeedForvardBlock(nn.Module):
        def __init__(self, in_channels, scale_factor):

            super().__init__()
            # Your code vvv
            self.Norm1 = LayerNorm2d(in_channels)

            out_channels = scale_factor * in_channels

            self.f1 = nn.Conv2d(in_channels=in_channels, out_channels=2 * out_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=1, bias=True)

            self.non_linear = nn.ReLU()

            self.f2 = nn.Conv2d(in_channels=out_channels, out_channels=in_channels, kernel_size=1, stride=1, groups=1, bias=False)
            # Your code ^^^

        def forward(self, x):

            # Your code vvv
            norm = self.Norm1(x)

            qq = self.f1(norm)

            q1, q2 = torch.chunk(qq, 2, dim=1)

            q1 = self.non_linear(q1)

            q = q1 * q2

            shift = self.f2(q)

            return x + shift
            # Your code ^^^
############################################################################################################################################
    class TransformerBlock(nn.Module):
        def __init__(self,  in_channels, pixelWiseHeads=3, dWiseHeads=1, feed_forvard_scale_factor = 3):

            super().__init__()
            # Your code vvv
            self.a = MultiHeadAttantion(in_channels, pixelWiseHeads, dWiseHeads)
            self.f = FeedForvardBlock(in_channels, feed_forvard_scale_factor)
            # Your code ^^^

        def forward(self, x):
            # Your code vvv
            x = self.a(x)

            x = self.f(x)

            return x
            # Your code ^^^
############################################################################################################################################
    class MyDownBlock(nn.Module):
        def __init__(self, in_channels):
            super().__init__()
            # Your code vvv

            out_channels = 2 * in_channels

            self.f1 = nn.Conv2d(in_channels=in_channels, out_channels=2 * out_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=1, groups=1, bias=True)

            self.non_linear = nn.ReLU()

            self.f2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1, padding_mode='reflect', stride=2, groups=out_channels, bias=False)

            # Your code ^^^

        def forward(self, x):
            # Your code vvv

            qq = self.f1(x)

            q1, q2 = torch.chunk(qq, 2, dim=1)

            q1 = self.non_linear(q1)

            q = q1 * q2

            x = self.f2(q)

            return x
            # Your code ^^^
############################################################################################################################################
    class MyUpBlock(nn.Module):
        def __init__(self, channels):
            super().__init__()
            # Your code vvv
            self.s_conv = nn.Conv2d(channels, 2 * channels, kernel_size=1, bias=True)
            self.s_pixel_shuffle = nn.PixelShuffle(2)
            # Your code ^^^

        def forward(self, x):
            # Your code vvv
            x = self.s_conv(x)

            x = self.s_pixel_shuffle(x)

            return x
            # Your code ^^^
############################################################################################################################################


    model = MyGeneralizedUNet(TransformerBlock, MyDownBlock, MyUpBlock)
    if model_path is not None:
        model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu'))['model_state_dict']) # Пример загрузки модели из файла
    else:
        # Пример обучения модели
        model = model.to(device)

        criterion = PSNRLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        use_grad_clip = True
        scheduler = get_scheduler(optimizer)

        train_model(model, train_dataloader, optimizer, criterion, scheduler, num_epochs=2000, checkpoints_path='./checkpoints/my_cool_model', use_grad_clip=use_grad_clip)
    # Your code ^^^
    return model.to(device)

Протестируем нашу модель на GoPro:

In [ ]:
my_cool_model_path = os.path.join(WSD, './my_cool_model.pth')

In [ ]:
model = get_my_cool_model()
model.eval()
test_model(model, device, test_dataloader)

In [ ]:
torch.save({
        'model_state_dict': model.state_dict()
        }, my_cool_model_path)

# Ваши впечатления

Пожалуйста, поделитесь своими мыслями о задании.
Понравилось ли оно вам или нет?

Если да/нет, то почему?

**Место для вашего ответа**